# 1. Insertion Sort Algorithm (Shift-and-insert)

In [1]:
from typing import List

class Pair:
    def __init__(self, key: int, value: str):
        self.key = key
        self.value = value

    def __repr__(self) -> str:
        return f"Pair(key={self.key}, value={self.value!r})"

def insertion_sort(pairs: List[Pair]) -> List[Pair]:
    for i in range(1, len(pairs)):
        current = pairs[i]
        j = i - 1
        # Stable: maintain relative order of equal elements. 
        # This is not in specification of insertion sort, but it is still an invariant of the algorithm?
        while j >= 0 and pairs[j].key > current.key: 
            
            pairs[j + 1] = pairs[j]
            j -= 1
        pairs[j + 1] = current
    return pairs

Using `>` rather than `>=` preserves the relative order of equal-key objects,
so this implementation is stable.

$$\forall x,y.\;
Processed(x)\land Processed(y)\land
OriginalBefore(x,y)\land
KeyEqual(x,y)
\Rightarrow Before(x,y)$$

In [2]:
pair = Pair(5, "apple")
repr(pair)

"Pair(key=5, value='apple')"

In [3]:
def get_keys(pairs: List[Pair]) -> List[int]:
    return [pair.key for pair in pairs]

def get_values(pairs: List[Pair]) -> List[str]:
    return [pair.value for pair in pairs]

assert get_keys(insertion_sort([])) == []
assert get_keys(insertion_sort([Pair(1, "a")])) == [1]
assert get_keys(
    insertion_sort([
        Pair(5, "apple"),
        Pair(2, "banana"),
        Pair(9, "cherry"),
    ])
) == [2, 5, 9]
assert get_keys(
    insertion_sort([
        Pair(4, "a"),
        Pair(3, "b"),
        Pair(2, "c"),
        Pair(1, "d"),
    ])
) == [1, 2, 3, 4]

In [4]:
result = insertion_sort([
    Pair(2, "first"),
    Pair(1, "middle"),
    Pair(2, "second"),
])
assert get_keys(result) == [1, 2, 2]
assert get_values(result) == ["middle", "first", "second"]

print("All insertion-sort tests passed.")

All insertion-sort tests passed.


# 2. Record raw snapshots

In [5]:
from typing import Tuple


def insertion_sort_with_snapshots(
    pairs: List[Pair],
) -> Tuple[List[Pair], List[List[Pair]]]:

    if not pairs:
        return pairs, []

    snapshots: List[List[Pair]] = []

    # Record the initial state before the first outer-loop iteration
    snapshots.append(pairs.copy())


    for i in range(1, len(pairs)):
        # INV + (i = 1, 2, ...., len(pairs)))
        current = pairs[i]
        j = i - 1
        # Stable: maintain relative order of equal elements. 
        # This is not in specification of insertion sort, but it is still an invariant of the algorithm?
        while j >= 0 and pairs[j].key > current.key:         
            pairs[j + 1] = pairs[j]
            j -= 1
        # snapshots.append(pairs.copy())
        pairs[j + 1] = current

        # Record the state after each outer-loop iteration
        snapshots.append(pairs.copy())

    return pairs, snapshots

In [6]:
# Quick Test
pairs, snapshots = insertion_sort_with_snapshots([
        Pair(5, "apple"),
        Pair(2, "banana"),
        Pair(9, "cherry")
    ])
snapshots

[[Pair(key=5, value='apple'),
  Pair(key=2, value='banana'),
  Pair(key=9, value='cherry')],
 [Pair(key=2, value='banana'),
  Pair(key=5, value='apple'),
  Pair(key=9, value='cherry')],
 [Pair(key=2, value='banana'),
  Pair(key=5, value='apple'),
  Pair(key=9, value='cherry')]]

In [7]:
example = [
    Pair(5, "apple"),
    Pair(2, "banana"),
    Pair(9, "cherry"),
]

sorted_pairs, snapshots = insertion_sort_with_snapshots(example)

snapshot_keys = [get_keys(snapshot) for snapshot in snapshots]

# assert snapshot_keys == [
#     [5, 2, 9],  # initial state: prefix length 1
#     [2, 5, 9],  # after inserting key 2: prefix length 2
#     [2, 5, 9],  # after inserting key 9: prefix length 3
# ]

# assert get_keys(sorted_pairs) == [2, 5, 9]
# assert snapshots[0] is not snapshots[1]
# assert snapshots[1] is not snapshots[2]

print(snapshot_keys)

[[5, 2, 9], [2, 5, 9], [2, 5, 9]]


In [8]:
[id(pair) for pair in example]

[4425808528, 4425808672, 4425806032]

In [9]:
[(prefix_length, current_order)for prefix_length, current_order in enumerate(
        snapshots,
        start=1,
    )]

[(1,
  [Pair(key=5, value='apple'),
   Pair(key=2, value='banana'),
   Pair(key=9, value='cherry')]),
 (2,
  [Pair(key=2, value='banana'),
   Pair(key=5, value='apple'),
   Pair(key=9, value='cherry')]),
 (3,
  [Pair(key=2, value='banana'),
   Pair(key=5, value='apple'),
   Pair(key=9, value='cherry')])]

## 2.1 Stable Object Identity

In [10]:
pairs = [
    Pair(5, "apple"),
    Pair(2, "banana"),
    Pair(9, "cherry"),
]

object_names = {
    id(pair): f"item_{index}"
    for index, pair in enumerate(pairs)
}

keys = {
    object_names[id(pair)]: pair.key
    for pair in pairs
}

values = {
    object_names[id(pair)]: pair.value
    for pair in pairs
}

initial_order = tuple(object_names[id(pair)] for pair in pairs)

print(initial_order)
print(keys)
print(values)

('item_0', 'item_1', 'item_2')
{'item_0': 5, 'item_1': 2, 'item_2': 9}
{'item_0': 'apple', 'item_1': 'banana', 'item_2': 'cherry'}


## 2.2 Define structured snapshot

In [11]:
from dataclasses import dataclass
from typing import Dict, Tuple

@dataclass(frozen=True)
class InsertionSortSnapshot:
    outer_index: int # length of the current sorted prefix
    order: Tuple[str, ...] # the order of the elements in the list at this point in time, e.g., ("item_1", "item_0", "item_2")
    processed: frozenset[str] # objects currently belonging to that sorted prefix, e.g., frozenset({"item_0", "item_1"})
    keys: Dict[str, int] # a mapping from each element to its key, e.g., {"item_0": 5, "item_1": 2, "item_2": 9}
    values: Dict[str, str] # a mapping from each element to its value, e.g., {"item_0": "apple", "item_1": "banana", "item_2": "cherry"}

## 2.3 Tracer

Convert raw snapshots to structured snapshots

In [12]:
def trace_insertion_sort(pairs: List[Pair]) -> Tuple[List[Pair], Tuple[InsertionSortSnapshot, ...]]:

    working = list(pairs)

    object_ids = [id(pair) for pair in working]

    if len(set(object_ids)) != len(working):
        raise ValueError("Each input position must contain a distinct Pair object.")

    # Stable names are assigned according to the original input positions.
    object_names = {
        id(pair): f"item_{index}" 
        for index, pair in enumerate(working)
    }

    keys = {
        object_names[id(pair)]: pair.key
        for pair in working
    }

    values = {
        object_names[id(pair)]: pair.value
        for pair in working
    }

    sorted_pairs, raw_snapshots = insertion_sort_with_snapshots(working)

    snapshots: List[InsertionSortSnapshot] = []

    for prefix_length, current_order in enumerate(
        raw_snapshots,
        start=1,
    ):
        # TODO 1:
        # Convert the current Pair order into stable object names.
        named_order = tuple(object_names[id(pair)] for pair in current_order)

        # TODO 2:
        # The first prefix_length objects have been processed.
        processed = frozenset(named_order[:prefix_length])

        # TODO 3:
        # Construct and append an InsertionSortSnapshot.
        snapshot = InsertionSortSnapshot(
            outer_index=prefix_length,
            order=named_order,
            processed=processed,
            keys=dict(keys),
            values=dict(values),
        )

        snapshots.append(snapshot)

    return sorted_pairs, tuple(snapshots)

In [13]:
original = [
    Pair(5, "apple"),
    Pair(2, "banana"),
    Pair(9, "cherry"),
]

sorted_pairs, snapshots = trace_insertion_sort(original)

assert get_keys(original) == [5, 2, 9]
assert get_keys(sorted_pairs) == [2, 5, 9]

# assert [snapshot.order for snapshot in snapshots] == [
#     ("item_0", "item_1", "item_2"),  # initial state: prefix length 1
#     ("item_1", "item_0", "item_2"),  # after inserting key 2: prefix length 2
#     ("item_1", "item_0", "item_2"),  # after inserting key 9: prefix length 3
# ]

# assert [snapshot.processed for snapshot in snapshots] == [
#     frozenset({"item_0"}),  # initial state: prefix length 1
#     frozenset({"item_0", "item_1"}),  # after inserting key 2: prefix length 2
#     frozenset({"item_0", "item_1", "item_2"}),  # after inserting key 9: prefix length 3
# ]

# assert snapshots[1].keys == {
#     "item_0": 5,
#     "item_1": 2,
#     "item_2": 9,
# }

for snapshot in snapshots:
    print(
        "prefix_length =", snapshot.outer_index,
        "order =", snapshot.order,
        "processed =", snapshot.processed,
    )

prefix_length = 1 order = ('item_0', 'item_1', 'item_2') processed = frozenset({'item_0'})
prefix_length = 2 order = ('item_1', 'item_0', 'item_2') processed = frozenset({'item_0', 'item_1'})
prefix_length = 3 order = ('item_1', 'item_0', 'item_2') processed = frozenset({'item_0', 'item_2', 'item_1'})


# 3. Build the Relational Model

## 3.1 Adapt to `RelationalSnapshot`

In [14]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from synthesis.inference_lib.relational import RelationalSnapshot

def to_relational_snapshot(snapshot: InsertionSortSnapshot) -> RelationalSnapshot:

    return RelationalSnapshot(
        # TODO 1:
        # The complete stable object universe
        objects=tuple(snapshot.keys.keys()),

        # TODO 2:
        # Preserve the algorithm-specific state
        payload=snapshot,
    )

In [15]:
relational_snapshot = to_relational_snapshot(snapshots[1])

assert relational_snapshot.objects == ("item_0", "item_1", "item_2")

assert relational_snapshot.payload is snapshots[1]
assert relational_snapshot.constant_bindings == {}

print("objects:", relational_snapshot.objects)
print("current order:", relational_snapshot.payload.order)
print("processed:", relational_snapshot.payload.processed)

objects: ('item_0', 'item_1', 'item_2')
current order: ('item_1', 'item_0', 'item_2')
processed: frozenset({'item_0', 'item_1'})


In [16]:
relational_snapshots = tuple(to_relational_snapshot(snapshot) for snapshot in snapshots)

assert len(relational_snapshots) == len(snapshots)
assert all(
    relational_snapshot.objects == ("item_0", "item_1", "item_2")
    for relational_snapshot in relational_snapshots
)

print(relational_snapshots[1])

RelationalSnapshot(objects=('item_0', 'item_1', 'item_2'), payload=InsertionSortSnapshot(outer_index=2, order=('item_1', 'item_0', 'item_2'), processed=frozenset({'item_0', 'item_1'}), keys={'item_0': 5, 'item_1': 2, 'item_2': 9}, values={'item_0': 'apple', 'item_1': 'banana', 'item_2': 'cherry'}), constant_bindings={})


In [17]:
tuple(snapshots[1].keys)

('item_0', 'item_1', 'item_2')

## 3.2 Define Concrete Relation Evaluators (Predicates)

Refer to Sec 1, if we don't define `OriginalBefore` predicate here, it will not be able to infer the stable property.

In [18]:
def processed(
    snapshot: RelationalSnapshot,
    arguments: Tuple[str, ...],
) -> bool:

    (item,) = arguments

    # TODO:
    # Is item in the processed prefix?

    return item in snapshot.payload.processed

In [19]:
second_state = relational_snapshots[1]

assert processed(second_state, ("item_0",)) is True
assert processed(second_state, ("item_1",)) is True
assert processed(second_state, ("item_2",)) is False

In [20]:
def before(
    snapshot: RelationalSnapshot,
    arguments: Tuple[str, ...],
) -> bool:

    left, right = arguments

    positions = {item: index for index, item in enumerate(snapshot.payload.order)}

    return positions[left] < positions[right]

In [21]:
assert before(second_state, ("item_1", "item_0")) is True
assert before(second_state, ("item_0", "item_1")) is False

# Strict order: no object is before itself.
assert before(second_state, ("item_0", "item_0")) is False

In [22]:
# assert before(relational_snapshots[0], ("item_0", "item_1")) is True
# assert before(relational_snapshots[1], ("item_0", "item_1")) is False

In [23]:
def key_le(
    snapshot: RelationalSnapshot,
    arguments: Tuple[str, ...],
) -> bool:

    left, right = arguments

    return snapshot.payload.keys[left] <= snapshot.payload.keys[right]

In [24]:
assert key_le(second_state, ("item_1", "item_0")) is True   # 2 <= 5
assert key_le(second_state, ("item_0", "item_1")) is False  # 5 <= 2
assert key_le(second_state, ("item_0", "item_0")) is True   # 5 <= 5

In [25]:
for state in relational_snapshots:
    assert key_le(state, ("item_1", "item_0")) is True

In [26]:
relation_results = {
    "Processed(item_1)": processed(second_state, ("item_1",)),
    "Processed(item_2)": processed(second_state, ("item_2",)),
    "Before(item_1,item_0)": before(
        second_state, ("item_1", "item_0")
    ),
    "KeyLE(item_1,item_0)": key_le(
        second_state, ("item_1", "item_0")
    ),
}

assert relation_results == {
    "Processed(item_1)": True,
    "Processed(item_2)": False,
    "Before(item_1,item_0)": True,
    "KeyLE(item_1,item_0)": True,
}

relation_results

{'Processed(item_1)': True,
 'Processed(item_2)': False,
 'Before(item_1,item_0)': True,
 'KeyLE(item_1,item_0)': True}

## 3.3 Describe Relations with `RelationSpec`

In [27]:
from synthesis.inference_lib.relational import (
    RelationSpec,
    RelationalDomain,
)

processed_spec = RelationSpec(
    name="Processed",       # symbolic predicate name
    arity=1,      # number of object arguments
    evaluator=processed,  # concrete Python function
)

before_spec = RelationSpec(
    name="Before",
    arity=2,
    evaluator=before,
)

key_le_spec = RelationSpec(
    name="KeyLE",
    arity=2,
    evaluator=key_le,
)

In [28]:
assert processed_spec.name == "Processed"
assert processed_spec.arity == 1
assert processed_spec.evaluator is processed

assert before_spec.name == "Before"
assert before_spec.arity == 2
assert before_spec.evaluator is before

assert key_le_spec.name == "KeyLE"
assert key_le_spec.arity == 2
assert key_le_spec.evaluator is key_le

In [29]:
assert processed_spec.evaluator(
    second_state,
    ("item_1",),
) is True

## 3.4 Build minimal `RelationalDomain`

In [30]:
domain = RelationalDomain(
    name="InsertionSortItem",
    relation_specs=(
        processed_spec,
        before_spec,
        key_le_spec,
    ),
    include_equality=True,
)

In [31]:
print("Object sort:", domain.object_sort) 
# object_sort is the Z3 type inhabited by insertion-sort objects.
# The concrete finite object universe is snapshot.objects.

for name, relation in domain.relations.items():
    print(name, "->", relation)

Object sort: InsertionSortItem
Processed -> Processed
Before -> Before
KeyLE -> KeyLE


In [32]:
x = domain.constant("x")
y = domain.constant("y")

# These are z3 expressions, not Python booleans.
print(domain.relation("Processed")(x))
print(domain.relation("Before")(x, y))
print(domain.relation("KeyLE")(x, y))

Processed(x)
Before(x, y)
KeyLE(x, y)


In [33]:
import z3

assert isinstance(
    domain.relation("Before")(x, y),
    z3.BoolRef,
)

In [34]:
assert domain.evaluate(
    "Processed",
    second_state,
    ("item_1",),
) is True

assert domain.evaluate(
    "Before",
    second_state,
    ("item_1", "item_0"),
) is True

assert domain.evaluate(
    "KeyLE",
    second_state,
    ("item_1", "item_0"),
) is True

In [35]:
try:
    domain.evaluate(
        "Processed",
        second_state,
        ("item_0", "item_1"),
    )
except ValueError as error:
    print(error)

Relation 'Processed' expects 1 arguments, got 2.


# 4. Add Domain Axioms  

`Before` models the current positional order and is a strict total order:

1. Irreflexive:
   $\forall x.\ \neg Before(x,x)$

2. Transitive:
   $\forall x,y,z.\ Before(x,y)\land Before(y,z)
   \Rightarrow Before(x,z)$

3. Total on distinct objects:
   $\forall x,y.\ x=y\lor Before(x,y)\lor Before(y,x)$

`KeyLE` models `key(x) <= key(y)` and is a total preorder:

4. Reflexive:
   $\forall x.\ KeyLE(x,x)$

5. Transitive:
   $\forall x,y,z.\ KeyLE(x,y)\land KeyLE(y,z)
   \Rightarrow KeyLE(x,z)$

6. Total:
   $\forall x,y.\ KeyLE(x,y)\lor KeyLE(y,x)$

In [36]:
def insertion_sort_axioms(domain: RelationalDomain) -> Tuple[z3.BoolRef, ...]:

    before_relation = domain.relation("Before")
    key_le_relation = domain.relation("KeyLE")

    axiom_x, axiom_y, axiom_z = z3.Consts(
        "axiom_x axiom_y axiom_z",
        domain.object_sort,
    )

    before_irreflexive = z3.ForAll(
        [axiom_x],
        z3.Not(before_relation(axiom_x, axiom_x))
    )

    before_transitive = z3.ForAll(
        [axiom_x, axiom_y, axiom_z],
        z3.Implies(
            z3.And(before_relation(axiom_x, axiom_y), before_relation(axiom_y, axiom_z)),
            before_relation(axiom_x, axiom_z)
        )
    )

    before_total = z3.ForAll(
        [axiom_x, axiom_y],
        z3.Or(
            before_relation(axiom_x, axiom_y),
            before_relation(axiom_y, axiom_x),
            axiom_x == axiom_y
        )
    )

    key_le_reflexive = z3.ForAll(
        [axiom_x],
        key_le_relation(axiom_x, axiom_x)
    )

    key_le_transitive = z3.ForAll(
        [axiom_x, axiom_y, axiom_z],
        z3.Implies(
            z3.And(key_le_relation(axiom_x, axiom_y), key_le_relation(axiom_y, axiom_z)),
            key_le_relation(axiom_x, axiom_z)
        )
    )

    key_le_total = z3.ForAll(
        [axiom_x, axiom_y],
        z3.Or(
            key_le_relation(axiom_x, axiom_y),
            key_le_relation(axiom_y, axiom_x)
        )
    )

    return (
        before_irreflexive,
        before_transitive,
        before_total,
        key_le_reflexive,
        key_le_transitive,
        key_le_total,
    )

In [37]:
domain = RelationalDomain(
    name="InsertionSortItem",
    relation_specs=(
        processed_spec,
        before_spec,
        key_le_spec,
    ),
    include_equality=True,
    axiom_builder=insertion_sort_axioms,
)

In [38]:
axioms = insertion_sort_axioms(domain)
assert len(axioms) == 6
for index, axiom in enumerate(axioms, start=1):
    print(f"Axiom {index}: {axiom}")

Axiom 1: ForAll(axiom_x, Not(Before(axiom_x, axiom_x)))
Axiom 2: ForAll([axiom_x, axiom_y, axiom_z],
       Implies(And(Before(axiom_x, axiom_y),
                   Before(axiom_y, axiom_z)),
               Before(axiom_x, axiom_z)))
Axiom 3: ForAll([axiom_x, axiom_y],
       Or(Before(axiom_x, axiom_y),
          Before(axiom_y, axiom_x),
          axiom_x == axiom_y))
Axiom 4: ForAll(axiom_x, KeyLE(axiom_x, axiom_x))
Axiom 5: ForAll([axiom_x, axiom_y, axiom_z],
       Implies(And(KeyLE(axiom_x, axiom_y),
                   KeyLE(axiom_y, axiom_z)),
               KeyLE(axiom_x, axiom_z)))
Axiom 6: ForAll([axiom_x, axiom_y],
       Or(KeyLE(axiom_x, axiom_y), KeyLE(axiom_y, axiom_x)))


In [39]:
solver = z3.Solver()
domain.add_axioms(solver)

consistency_result = solver.check()
print("Axiom consistency:", consistency_result)

assert consistency_result == z3.sat # six symbolic axioms are mutually consistent

Axiom consistency: sat


In [40]:
# Check evaluator consistency on observed snapshots

from itertools import product

# for state in relational_snapshots:
#     objects = state.objects

#     for x_name in objects:
#         # Before is irreflexive.
#         assert before(state, (x_name, x_name)) is False

#         # KeyLE is reflexive.
#         assert key_le(state, (x_name, x_name)) is True

#     for x_name, y_name in product(objects, repeat=2):
#         # Before is total on distinct objects.
#         assert before(state, (x_name, y_name)) or before(state, (y_name, x_name)) or x_name == y_name

#         # KeyLE is total.
#         assert key_le(state, (x_name, y_name)) or key_le(state, (y_name, x_name))

#     for x_name, y_name, z_name in product(objects, repeat=3):
#         # Before is transitive.
#         if before(state, (x_name, y_name)) and before(state, (y_name, z_name)):
#             assert before(state, (x_name, z_name))

#         # KeyLE is transitive.
#         if key_le(state, (x_name, y_name)) and key_le(state, (y_name, z_name)):
#             assert key_le(state, (x_name, z_name))

print("Observed snapshots satisfy all six domain properties.")

Observed snapshots satisfy all six domain properties.


# 5. Build the Symbolic Vocabulary

The sorted-prefix property compares two objects:

$$
Processed(x)\land Processed(y)\land Before(x,y)
\Rightarrow KeyLE(x,y)
$$

Therefore, relational inference needs two universally quantified variables,
so we choose `k = 2`.

`k` controls the number of objects that a learned formula can discuss
simultaneously. It does not limit the input array length.

In [41]:
from synthesis.inference_lib.relational import build_relational_vocabulary

universal_variables, vocabulary = build_relational_vocabulary(
    domain=domain,
    k=2,
)

print("Universal variables:")

for variable in universal_variables:
    print(" ", variable)

print("\nVocabulary:")

for index, atom in enumerate(vocabulary):
    print(f"{index}: {atom}")

Universal variables:
  ux1
  ux2

Vocabulary:
0: Processed(ux1)
1: Processed(ux2)
2: Before(ux1, ux2)
3: Before(ux2, ux1)
4: KeyLE(ux1, ux2)
5: KeyLE(ux2, ux1)
6: ux1 == ux2


In [42]:
assert [str(variable) for variable in universal_variables] == [
    "ux1",
    "ux2",
]

assert [str(atom) for atom in vocabulary] == [
    "Processed(ux1)",
    "Processed(ux2)",
    "Before(ux1, ux2)",
    "Before(ux2, ux1)",
    "KeyLE(ux1, ux2)",
    "KeyLE(ux2, ux1)",
    "ux1 == ux2",
]

assert len(vocabulary) == 7

# 6. Ground Concrete Truth Rows

A vocabulary contains symbolic atoms involving `ux1` and `ux2`.
Grounding assigns concrete objects to those variables and evaluates every atom.

We use the second insertion-sort snapshot:

- current order: `item_1, item_0, item_2`
- processed objects: `item_1, item_0`
- keys: `item_1.key = 2`, `item_0.key = 5`

The assignment is:

- `ux1 = item_1`
- `ux2 = item_0`

In [43]:
grounding_state = relational_snapshots[1]
assignment = ("item_1", "item_0")

print("Order:", grounding_state.payload.order)
print("Processed:", grounding_state.payload.processed)
print("Assignment:", assignment)

Order: ('item_1', 'item_0', 'item_2')
Processed: frozenset({'item_0', 'item_1'})
Assignment: ('item_1', 'item_0')


In [44]:
expected_row = (
    True,
    True,
    True,
    False,
    True,
    False,
    False,
)

from synthesis.inference_lib.relational import (
    ground_relational_row,
)

grounded_row = ground_relational_row(
    domain=domain,
    snapshot=grounding_state,
    vocabulary=vocabulary,
    universal_variables=universal_variables,
    assignment=assignment,
)

assert grounded_row == expected_row

for atom, value in zip(vocabulary, grounded_row):
    print(f"{str(atom):24s} -> {value}")

Processed(ux1)           -> True
Processed(ux2)           -> True
Before(ux1, ux2)         -> True
Before(ux2, ux1)         -> False
KeyLE(ux1, ux2)          -> True
KeyLE(ux2, ux1)          -> False
ux1 == ux2               -> False


In [45]:
same_object_row = ground_relational_row(
    domain=domain,
    snapshot=grounding_state,
    vocabulary=vocabulary,
    universal_variables=universal_variables,
    assignment=("item_0", "item_0"),
)

assert same_object_row == (
    True,
    True,
    False,
    False,
    True,
    True,
    True,
)

for atom, value in zip(vocabulary, same_object_row):
    print(f"{str(atom):24s} -> {value}")

Processed(ux1)           -> True
Processed(ux2)           -> True
Before(ux1, ux2)         -> False
Before(ux2, ux1)         -> False
KeyLE(ux1, ux2)          -> True
KeyLE(ux2, ux1)          -> True
ux1 == ux2               -> True


# 7. Build the Training Corpus

every snapshot × every ordered assignment of universal variables

In [46]:
walkthrough_groundings = []

for snapshot_index, state in enumerate(relational_snapshots):

    assignments = product(state.objects, repeat=len(universal_variables))

    for assignment in assignments:
        row = ground_relational_row(
            domain=domain,
            snapshot=state,
            vocabulary=vocabulary,
            universal_variables=universal_variables,
            assignment=assignment,
        )

        walkthrough_groundings.append((snapshot_index, assignment, row))

In [47]:
assert len(walkthrough_groundings) == 3 * 3 * 3  # 3 snapshots, 3 objects, 3 objects

print("Raw grounded rows:", len(walkthrough_groundings))

for snapshot_index, assignment, row in walkthrough_groundings[:5]:
    print(
        "snapshot =", snapshot_index,
        "assignment =", assignment,
        "row =", row,
    )

Raw grounded rows: 27
snapshot = 0 assignment = ('item_0', 'item_0') row = (True, True, False, False, True, True, True)
snapshot = 0 assignment = ('item_0', 'item_1') row = (True, False, True, False, False, True, False)
snapshot = 0 assignment = ('item_0', 'item_2') row = (True, False, True, False, True, False, False)
snapshot = 0 assignment = ('item_1', 'item_0') row = (False, True, False, True, True, False, False)
snapshot = 0 assignment = ('item_1', 'item_1') row = (False, False, False, False, True, True, True)


In [48]:
distinct_manual_rows = frozenset(
    row
    for _, _, row in walkthrough_groundings
)

print("Raw grounded rows:", len(walkthrough_groundings))
print("Distinct truth rows:", len(distinct_manual_rows))

Raw grounded rows: 27
Distinct truth rows: 10


In [49]:
assert len(distinct_manual_rows) == 10
assert grounded_row in distinct_manual_rows
assert same_object_row in distinct_manual_rows

In [50]:
from synthesis.inference_lib.relational import (
    build_relational_dataset,
)

walkthrough_rows = build_relational_dataset(
    domain=domain,
    snapshots=relational_snapshots,
    vocabulary=vocabulary,
    universal_variables=universal_variables,
)

assert isinstance(walkthrough_rows, frozenset)
assert walkthrough_rows == distinct_manual_rows
assert len(walkthrough_rows) == 10

print("Walkthrough snapshots:", len(relational_snapshots))
print("Distinct walkthrough truth rows:", len(walkthrough_rows))

Walkthrough snapshots: 3
Distinct walkthrough truth rows: 10


Diversify

In [51]:
def pairs_from_keys(
    keys: Tuple[int, ...],
    case_name: str,
) -> List[Pair]:
    return [
        Pair(
            key=key,
            value=f"{case_name}_value_{index}",
        )
        for index, key in enumerate(keys)
    ]

In [52]:
duplicate_pairs = pairs_from_keys(
    (2, 1, 2),
    "duplicate",
)

assert get_keys(duplicate_pairs) == [2, 1, 2]
assert get_values(duplicate_pairs) == ["duplicate_value_0", "duplicate_value_1", "duplicate_value_2"]

assert len({id(pair) for pair in duplicate_pairs}) == 3

In [53]:
training_key_sequences = (
    (5, 2, 9),       # walkthrough
    (1,),            # singleton
    (1, 2),          # sorted pair
    (2, 1),          # reversed pair
    (1, 2, 3),       # already sorted
    (3, 2, 1),       # fully reversed
    (1, 3, 2),       # middle insertion
    (3, 4, 1),       # smallest item inserted from the end
    (2, 1, 2),       # duplicate keys
    (2, 2, 2),       # all keys equal
    (4, 1, 3, 2),    # longer mixed case
)

training_snapshots_list = []

for case_index, keys in enumerate(training_key_sequences):

    case_pairs = pairs_from_keys(keys, f"case_{case_index}")
    sorted_case, case_snapshots = trace_insertion_sort(case_pairs)

    assert get_keys(sorted_case) == sorted(keys)

    training_snapshots_list.extend(to_relational_snapshot(snapshot) for snapshot in case_snapshots)

training_snapshots = tuple(training_snapshots_list)

assert len(training_snapshots) == 30

print("Training snapshots:", len(training_key_sequences), "cases,", len(training_snapshots), "snapshots total.")

Training snapshots: 11 cases, 30 snapshots total.


In [54]:
def has_sorted_processed_prefix(
    snapshot: RelationalSnapshot,
) -> bool:

    state = snapshot.payload

    prefix = state.order[:state.outer_index]

    if state.processed != frozenset(prefix):
        return False

    prefix_keys = [
        state.keys[item] for item in prefix
    ]

    return all(
        left_key <= right_key
        for left_key, right_key in zip(prefix_keys, prefix_keys[1:])
    )

assert all(
    has_sorted_processed_prefix(snapshot)
    for snapshot in training_snapshots
)

print("Every training snapshot has a sorted processed prefix.")

Every training snapshot has a sorted processed prefix.


In [55]:
training_rows = build_relational_dataset(
    domain=domain,
    snapshots=training_snapshots,
    vocabulary=vocabulary,
    universal_variables=universal_variables,
)

print("Training snapshots:", len(training_snapshots))
print("Distinct training truth rows:", len(training_rows))

assert len(training_snapshots) == 30
assert len(training_rows) == 18

# Everything observed in the walkthrough should still be present.
assert walkthrough_rows.issubset(training_rows)

Training snapshots: 30
Distinct training truth rows: 18


# 8. Call the Core API

In [56]:
from synthesis.inference_lib.relational import (
    RelationalInferenceResult,
    infer_relational_invariants,
)

result = infer_relational_invariants(
    domain=domain,
    snapshots=training_snapshots,
    k=2,
    verbose=False, # only suppresses extensive backend diagnostics. It does not change the inference result.
)

In [57]:
assert isinstance(result, RelationalInferenceResult)

print("Universal variables:", len(result.universal_variables))
print("Vocabulary size:", len(result.vocabulary))
print("Distinct truth rows:", len(result.truth_rows))
print("Retained clauses:", len(result.clauses))

Universal variables: 2
Vocabulary size: 7
Distinct truth rows: 18
Retained clauses: 2


In [58]:
assert [
    str(variable)
    for variable in result.universal_variables
] == [
    str(variable)
    for variable in universal_variables
]

assert [
    str(atom)
    for atom in result.vocabulary
] == [
    str(atom)
    for atom in vocabulary
]

assert result.truth_rows == training_rows

print("Manual construction matches the core API.")

Manual construction matches the core API.


In [59]:
print("Result fields:")

print(
    "universal_variables:",
    result.universal_variables,
)

print(
    "vocabulary:",
    result.vocabulary,
)

print(
    "truth-row count:",
    len(result.truth_rows),
)

print(
    "clause count:",
    len(result.clauses),
)

Result fields:
universal_variables: (ux1, ux2)
vocabulary: (Processed(ux1), Processed(ux2), Before(ux1, ux2), Before(ux2, ux1), KeyLE(ux1, ux2), KeyLE(ux2, ux1), ux1 == ux2)
truth-row count: 18
clause count: 2


In [60]:
for index, clause in enumerate(
    result.clauses,
    start=1,
):
    print(f"\nClause {index}")
    print("expression:")
    print(clause.expr)
    print("target predicate:", clause.target_predicate)
    print("learned via:", clause.learned_via)


Clause 1
expression:
ForAll([ux1, ux2],
       Or(Before(ux2, ux1),
          Processed(ux1),
          Not(Processed(ux2))))
target predicate: Processed(ux1)
learned via: phi

Clause 2
expression:
ForAll([ux1, ux2],
       Or(Not(Processed(ux1)),
          Before(ux1, ux2),
          KeyLE(ux2, ux1)))
target predicate: Processed(ux1)
learned via: phi_prime


Clause 1 is equivalent to (This is INCORRECT!)
$$\forall x,y.\;
Processed(x)\land Before(y,x)
\Rightarrow Processed(y)$$

Clause 2 is equivalent to (This is INCORRECT!)
$$\forall x,y.\;
Processed(x)\land Before(y,x)
\Rightarrow KeyLE(y,x)$$
with which it is easy to prove the target property is satisfied.

# 9. Interpret and Validate Results

The desired property states that if two processed objects occur in the order
`x` before `y`, then `x.key <= y.key`:

$$
\forall x,y.\;
Processed(x)\land Processed(y)\land Before(x,y)
\Rightarrow KeyLE(x,y)
$$

In [61]:
def sorted_prefix_target(
    domain: RelationalDomain,
) -> z3.BoolRef:
    target_x, target_y = z3.Consts(
        "target_x target_y",
        domain.object_sort,
    )

    processed_relation = domain.relation("Processed")
    before_relation = domain.relation("Before")
    key_le_relation = domain.relation("KeyLE")

    return z3.ForAll(
        [target_x, target_y],
        z3.Implies(
            z3.And(
                processed_relation(target_x),
                processed_relation(target_y),
                before_relation(target_x, target_y),
            ),
            key_le_relation(target_x, target_y),
        ),
    )

target = sorted_prefix_target(domain)

print("Sorted-prefix target:")
print(target)

assert isinstance(target, z3.BoolRef)

Sorted-prefix target:
ForAll([target_x, target_y],
       Implies(And(Processed(target_x),
                   Processed(target_y),
                   Before(target_x, target_y)),
               KeyLE(target_x, target_y)))


In [62]:
axioms_only_solver = z3.Solver()

domain.add_axioms(axioms_only_solver)
axioms_only_solver.add(z3.Not(target))

axioms_only_result = axioms_only_solver.check()

print(
    "Axioms alone entail sorted-prefix:",
    axioms_only_result,
)

assert axioms_only_result == z3.sat

Axioms alone entail sorted-prefix: sat


In [63]:
invariant_only_solver = z3.Solver()

invariant_only_solver.add(result.invariant)
invariant_only_solver.add(z3.Not(target))

invariant_only_result = invariant_only_solver.check()

print(
    "Learned invariant alone entails sorted-prefix:",
    invariant_only_result,
)

assert invariant_only_result == z3.sat

Learned invariant alone entails sorted-prefix: sat


In [64]:
entailment_solver = z3.Solver()

domain.add_axioms(entailment_solver)
entailment_solver.add(result.invariant)
entailment_solver.add(z3.Not(target))

entailment_result = entailment_solver.check()

print(
    "Axioms and learned invariant entail target:",
    entailment_result,
)

assert entailment_result == z3.unsat

Axioms and learned invariant entail target: unsat


In [65]:
assert axioms_only_result == z3.sat
assert invariant_only_result == z3.sat
assert entailment_result == z3.unsat

print("""
Axioms only:             counterexample exists
Learned invariant only:  counterexample exists
Axioms + invariant:      no counterexample exists
""")


Axioms only:             counterexample exists
Learned invariant only:  counterexample exists
Axioms + invariant:      no counterexample exists



In [66]:
for i in range(1, 4):
    i = 5
    print(i)


5
5
5
